In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "ma_walk_forward"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:.4f}",
)

plt.rcParams[
    "axes.unicode_minus"
] = False


%load_ext autoreload
%autoreload 2
from parameter_sensitivity import (
    build_ma_parameter_grid,
)

from walk_forward import (
    generate_walk_forward_windows,
    run_ma_walk_forward,
    run_fixed_parameter_validation,
    plot_walk_forward_metric,
    plot_selected_parameters,
)

In [ ]:
stock_list = [
    "000001",
    "000002",
    "300750",
    "600036",
    "600519",
]

fast_windows = [
    5,
    10,
    15,
    20,
    25,
    30,
    35,
    40,
]

slow_windows = [
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    110,
    120,
]

parameter_grid = build_ma_parameter_grid(
    fast_windows=fast_windows,
    slow_windows=slow_windows,
    min_gap=10,
)

print(
    "参数组合数量：",
    len(parameter_grid),
)

In [ ]:
test_windows = (
    generate_walk_forward_windows(
        start_date="2015-01-01",
        end_date="2024-12-31",
        train_months=36,
        test_months=12,
        step_months=12,
        window_type="rolling",
        include_partial_test=False,
    )
)

display(test_windows)

## 冒烟测试

In [ ]:
smoke_grid = [
    (5, 30),
    (10, 40),
    (15, 50),
    (20, 60),
]

smoke_result = run_ma_walk_forward(
    stock_list=[
        "000001",
        "600036",
    ],
    parameter_grid=smoke_grid,
    train_months=36,
    test_months=12,
    step_months=12,
    window_type="rolling",
    start_date=None,
    end_date="2024-12-31",
    commission_rate=0.0003,
    slippage_rate=0.0002,
    robustness_metric=(
        "avg_strategy_sharpe"
    ),
    stability_penalty=1.0,
    min_local_parameter_count=1,
)

assert not smoke_result[
    "windows"
].empty

assert not smoke_result[
    "selected_parameters"
].empty

assert not smoke_result[
    "test_detail"
].empty

assert not smoke_result[
    "test_summary"
].empty

assert (
    smoke_result[
        "selected_parameters"
    ]["fold_id"].nunique()
    == len(
        smoke_result["windows"]
    )
)

print(
    "Walk-Forward 冒烟测试通过"
)

## 正式测试

In [ ]:
walk_forward_result = (
    run_ma_walk_forward(
        stock_list=stock_list,
        parameter_grid=parameter_grid,
        train_months=36,
        test_months=12,
        step_months=12,
        window_type="rolling",
        start_date=None,
        end_date="2024-12-31",
        include_partial_test=False,
        commission_rate=0.0003,
        slippage_rate=0.0002,
        annual_risk_free_rate=0.0,
        trading_days=252,
        robustness_metric=(
            "avg_strategy_sharpe"
        ),
        stability_penalty=1.0,
        min_local_parameter_count=9,
    )
)

windows = walk_forward_result[
    "windows"
]

selected_parameters = (
    walk_forward_result[
        "selected_parameters"
    ]
)

parameter_frequency = (
    walk_forward_result[
        "parameter_frequency"
    ]
)

test_detail = walk_forward_result[
    "test_detail"
]

test_summary = walk_forward_result[
    "test_summary"
]

In [ ]:
selection_columns = [
    "fold_id",
    "train_start",
    "train_end",
    "test_start",
    "test_end",
    "selected_ma_param",
    "train_avg_strategy_sharpe",
    "train_avg_excess_annual_return",
    "train_return_win_rate",
    "train_local_mean",
    "train_local_std",
    "train_local_min",
    "train_robustness_score",
]

display(
    selected_parameters[
        selection_columns
    ]
)

In [ ]:
display(parameter_frequency)

In [ ]:
parameter_changes = (
    selected_parameters[
        [
            "fold_id",
            "selected_fast_window",
            "selected_slow_window",
            "selected_ma_param",
        ]
    ]
    .copy()
)

parameter_changes[
    "fast_change"
] = (
    parameter_changes[
        "selected_fast_window"
    ]
    .diff()
    .abs()
)

parameter_changes[
    "slow_change"
] = (
    parameter_changes[
        "selected_slow_window"
    ]
    .diff()
    .abs()
)

parameter_changes[
    "distance_to_10_40"
] = (
    (
        parameter_changes[
            "selected_fast_window"
        ]
        - 10
    ).abs()
    + (
        parameter_changes[
            "selected_slow_window"
        ]
        - 40
    ).abs()
)

display(parameter_changes)

In [ ]:
train_test_comparison = selected_parameters.merge(
    test_summary[
        [
            "fold_id",
            "selected_ma_param",
            "avg_strategy_sharpe",
            "avg_excess_annual_return",
            "return_win_rate",
            "avg_drawdown_improvement",
            "avg_trade_count",
        ]
    ],
    on=[
        "fold_id",
        "selected_ma_param",
    ],
    how="left",
)

train_test_comparison = (
    train_test_comparison.rename(
        columns={
            "avg_strategy_sharpe":
                "test_avg_strategy_sharpe",
            "avg_excess_annual_return":
                "test_avg_excess_annual_return",
            "return_win_rate":
                "test_return_win_rate",
            "avg_drawdown_improvement":
                "test_avg_drawdown_improvement",
            "avg_trade_count":
                "test_avg_trade_count",
        }
    )
)

train_test_comparison[
    "sharpe_change"
] = (
    train_test_comparison[
        "test_avg_strategy_sharpe"
    ]
    - train_test_comparison[
        "train_avg_strategy_sharpe"
    ]
)

train_test_comparison[
    "excess_return_change"
] = (
    train_test_comparison[
        "test_avg_excess_annual_return"
    ]
    - train_test_comparison[
        "train_avg_excess_annual_return"
    ]
)

display(
    train_test_comparison[
        [
            "fold_id",
            "selected_ma_param",
            "train_avg_strategy_sharpe",
            "test_avg_strategy_sharpe",
            "sharpe_change",
            "train_avg_excess_annual_return",
            "test_avg_excess_annual_return",
            "excess_return_change",
            "test_return_win_rate",
            "test_avg_drawdown_improvement",
        ]
    ]
)

In [ ]:
fixed_result = run_fixed_parameter_validation(
    stock_list=stock_list,
    parameters=[
        (10, 40),
        (20, 60),
        (25, 110),
    ],
    windows=windows,
    commission_rate=0.0003,
    slippage_rate=0.0002,
    annual_risk_free_rate=0.0,
    trading_days=252,
)

fixed_summary = fixed_result["summary"]

fixed_overall = (
    fixed_summary
    .groupby(
        "ma_param",
        as_index=False,
    )
    .agg(
        fold_count=(
            "fold_id",
            "nunique",
        ),
        avg_test_strategy_sharpe=(
            "avg_strategy_sharpe",
            "mean",
        ),
        median_test_strategy_sharpe=(
            "avg_strategy_sharpe",
            "median",
        ),
        avg_test_excess_return=(
            "avg_excess_annual_return",
            "mean",
        ),
        median_test_excess_return=(
            "avg_excess_annual_return",
            "median",
        ),
        positive_excess_fold_rate=(
            "avg_excess_annual_return",
            lambda values: (
                values > 0
            ).mean(),
        ),
        avg_return_win_rate=(
            "return_win_rate",
            "mean",
        ),
        avg_drawdown_improvement=(
            "avg_drawdown_improvement",
            "mean",
        ),
        avg_trade_count=(
            "avg_trade_count",
            "mean",
        ),
    )
)

display(
    fixed_overall.sort_values(
        "avg_test_strategy_sharpe",
        ascending=False,
    )
)

In [ ]:
dynamic_overall = pd.DataFrame(
    {
        "ma_param": [
            "dynamic_walk_forward"
        ],
        "fold_count": [
            test_summary[
                "fold_id"
            ].nunique()
        ],
        "avg_test_strategy_sharpe": [
            test_summary[
                "avg_strategy_sharpe"
            ].mean()
        ],
        "median_test_strategy_sharpe": [
            test_summary[
                "avg_strategy_sharpe"
            ].median()
        ],
        "avg_test_excess_return": [
            test_summary[
                "avg_excess_annual_return"
            ].mean()
        ],
        "median_test_excess_return": [
            test_summary[
                "avg_excess_annual_return"
            ].median()
        ],
        "positive_excess_fold_rate": [
            (
                test_summary[
                    "avg_excess_annual_return"
                ] > 0
            ).mean()
        ],
        "avg_return_win_rate": [
            test_summary[
                "return_win_rate"
            ].mean()
        ],
        "avg_drawdown_improvement": [
            test_summary[
                "avg_drawdown_improvement"
            ].mean()
        ],
        "avg_trade_count": [
            test_summary[
                "avg_trade_count"
            ].mean()
        ],
    }
)

overall_comparison = pd.concat(
    [
        fixed_overall,
        dynamic_overall,
    ],
    ignore_index=True,
)

display(
    overall_comparison.sort_values(
        "avg_test_strategy_sharpe",
        ascending=False,
    )
)